# ETTh1 exploratory data analysis

This notebook is a guided first look at the dataset used by both DLinear and PatchTST. It calls the tested project code for loading, validation, splitting, scaling, and window construction.

**How to use it:** select the project `.venv` Python kernel, then choose **Run → Run All Cells**. Read the short notes above each result and change plot ranges or variables freely.

## 1. Load the validated dataset

The loader checks timestamps, columns, missing values, and hourly spacing before returning any data.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from ts_project.data import (
    BENCHMARK_ROWS,
    FEATURE_COLUMNS,
    TRAIN_ROWS,
    VALIDATION_ROWS,
    build_window_datasets,
    prepare_etth1,
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (12, 4), "axes.titlesize": 12})

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "ETTh1.csv"
data = prepare_etth1(DATA_PATH)
raw = data.raw
benchmark = data.benchmark

print(f"Loaded {len(raw):,} rows from {raw.index.min()} to {raw.index.max()}")

## 2. Shape, columns, and first observations

`OT` is the transformer oil temperature. The other six variables describe different load measurements.

In [ ]:
overview = pd.DataFrame(
    {
        "value": [
            len(raw),
            len(raw.columns),
            raw.index.min(),
            raw.index.max(),
            raw.index.inferred_freq,
            int(raw.isna().sum().sum()),
            int(raw.index.duplicated().sum()),
        ]
    },
    index=[
        "Rows",
        "Variables",
        "First timestamp",
        "Last timestamp",
        "Inferred frequency",
        "Missing values",
        "Duplicate timestamps",
    ],
)
display(overview)
display(raw.head())

## 3. Numerical summary

Compare each variable's center, spread, and range. Large differences in scale are why the models use training-fitted standardization.

In [ ]:
display(raw.describe().T.round(3))

## 4. The seven time series

These full-resolution plots reveal long trends, seasonal movement, sudden changes, and differing variable scales.

In [ ]:
axes = raw.plot(
    subplots=True,
    layout=(4, 2),
    figsize=(14, 12),
    sharex=True,
    legend=False,
    linewidth=0.7,
    title=list(raw.columns),
)
plt.suptitle("ETTh1 variables across the complete public file", y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

## 5. Official benchmark boundaries

The papers use the first 12 months for training, followed by 4 months for validation and 4 months for testing. The remaining public observations are not part of this standard benchmark.

In [ ]:
train_end = benchmark.index[TRAIN_ROWS]
validation_end = benchmark.index[TRAIN_ROWS + VALIDATION_ROWS]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(benchmark.index, benchmark["OT"], linewidth=0.8, color="#243b53")
ax.axvspan(benchmark.index[0], train_end, alpha=0.14, color="#2a9d8f", label="Train")
ax.axvspan(train_end, validation_end, alpha=0.16, color="#e9c46a", label="Validation")
ax.axvspan(validation_end, benchmark.index[-1], alpha=0.14, color="#e76f51", label="Test")
ax.set(title="Oil temperature (OT) and chronological benchmark splits", ylabel="OT")
ax.legend(ncols=3)
plt.show()

split_summary = pd.DataFrame(
    {
        name: {
            "rows": len(part),
            "start": part.index.min(),
            "end": part.index.max(),
            "OT mean": part["OT"].mean(),
            "OT std": part["OT"].std(),
        }
        for name, part in ((name, data.split(name)) for name in ("train", "validation", "test"))
    }
).T
display(split_summary)

## 6. Distributions and relationships

Histograms show whether variables are symmetric, skewed, or multimodal. The correlation matrix shows linear relationships, but correlation does not imply causation.

In [ ]:
raw.hist(bins=50, figsize=(14, 10), color="#3a86ff", edgecolor="white")
plt.suptitle("Variable distributions", y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

correlation = benchmark.corr()
fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(correlation, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(FEATURE_COLUMNS)), FEATURE_COLUMNS, rotation=45, ha="right")
ax.set_yticks(range(len(FEATURE_COLUMNS)), FEATURE_COLUMNS)
for row in range(len(FEATURE_COLUMNS)):
    for column in range(len(FEATURE_COLUMNS)):
        ax.text(column, row, f"{correlation.iloc[row, column]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(image, ax=ax, label="Pearson correlation")
ax.set_title("Correlation within the benchmark period")
plt.tight_layout()
plt.show()

## 7. Hourly and weekly seasonality

Averaging oil temperature by hour and weekday gives a simple view of recurring calendar patterns.

In [ ]:
train = data.split("train")
hourly_ot = train["OT"].groupby(train.index.hour).mean()
weekday_ot = train["OT"].groupby(train.index.dayofweek).mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
hourly_ot.plot(ax=axes[0], marker="o", color="#3a86ff")
axes[0].set(title="Mean training OT by hour", xlabel="Hour of day", ylabel="Mean OT")
weekday_ot.plot(ax=axes[1], kind="bar", color="#8338ec")
axes[1].set_xticklabels(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], rotation=0)
axes[1].set(title="Mean training OT by weekday", xlabel="Weekday", ylabel="Mean OT")
plt.tight_layout()
plt.show()

## 8. What one forecasting example looks like

The first validation example uses the previous 336 hours as input and asks the model to forecast the next 96 hours. Notice that the target starts only after the input ends.

In [ ]:
windows = build_window_datasets(data, input_length=336, prediction_length=96)
origin = windows["validation"].origin_at(0)
input_slice = benchmark.iloc[origin - 336 : origin]
target_slice = benchmark.iloc[origin : origin + 96]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(input_slice.index, input_slice["OT"], label="336-hour input", color="#3a86ff")
ax.plot(target_slice.index, target_slice["OT"], label="96-hour target", color="#e63946")
ax.axvline(target_slice.index[0], color="black", linestyle="--", linewidth=1, label="Forecast origin")
ax.set(title="One validation forecasting window", ylabel="OT")
ax.legend()
plt.show()

for name, dataset in windows.items():
    sample_input, sample_target = dataset[0]
    print(
        f"{name:>10}: {len(dataset):,} windows | "
        f"input {tuple(sample_input.shape)} -> target {tuple(sample_target.shape)}"
    )

## Questions to think about

1. Which variables appear most strongly related?
2. Do the validation and test periods look statistically similar to training?
3. Which repeating patterns might help a forecasting model?
4. Are there abrupt changes or unusual regions that may make forecasting difficult?
5. After changing `prediction_length` to 192 or 336, how does the forecasting task feel different?

Your observations here can later guide the meaningful improvement we design.